<a href="https://colab.research.google.com/github/ctrlcoded/Resolving-Idioms-to-Improve-the-Machine-Translation-in-Indic-Languages/blob/main/m2m_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install transformers[sentencepiece] sacrebleu pandas openpyxl torch

In [5]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import compute_engine
from google.colab import auth
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

In [6]:
import pandas as pd

# Open the sheet by name
spreadsheet = gc.open('synthetic_data')
worksheet = spreadsheet.get_worksheet(0) # Gets the first tab

# Get all values into a list of lists, then convert to DataFrame
data = worksheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

print(f"Loaded {len(df)} rows.")

Loaded 1999 rows.


In [ ]:
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from tqdm import tqdm

model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name).to("cuda")
tokenizer.src_lang = "hi"

def translate_batch(sentences, batch_size=16):
    results = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to("cuda")
        with torch.no_grad():
            generated_tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.get_lang_id("en"))
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        results.extend(decoded)
    return results

In [ ]:

hindi_list = df['Sentence'].tolist()

print("Starting translation of 2000 sentences...")
translations = translate_batch(hindi_list)


output_column = [['M2M_Baseline_Output']] + [[t] for t in translations]

col_index = len(df.columns) + 1
range_to_update = f"{gspread.utils.rowcol_to_a1(1, col_index)}:{gspread.utils.rowcol_to_a1(len(output_column), col_index)}"


worksheet.update(range_to_update, output_column)


In [ ]:
import sacrebleu
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download('wordnet')

def calculate_standard_metrics(hypotheses, references):
    # BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])

    # METEOR (calculated per sentence and averaged)
    meteor_total = sum(meteor_score([ref.split()], hyp.split())
                       for ref, hyp in zip(references, hypotheses))
    avg_meteor = meteor_total / len(hypotheses)

    return bleu.score, avg_meteor

In [ ]:
def calculate_ler(df, sentence_col, output_col):
    """
    df: your dataframe
    sentence_col: Hindi sentence
    output_col: Model output
    """
    literal_count = 0
    # Dictionary of literal keywords for each idiom
    # You can generate this mapping for your 200 idioms
    literal_keywords = {
        "आँख का तारा": ["eye", "star"],
        "नाक कटना": ["nose", "cut"],
        "टोकरी उलटना": ["basket", "flip", "overturn"]
    }

    for _, row in df.iterrows():
        # Check if any literal word appears in the English output
        for idiom, words in literal_keywords.items():
            if idiom in row[sentence_col]:
                if any(word in row[output_col].lower() for word in words):
                    literal_count += 1
                    break

    return (literal_count / len(df)) * 100

In [ ]:
def get_idiomatic_accuracy_sample(df, n=100):
    sample = df.sample(n)
    print("Please mark 1 for Idiomatic (figurative) and 0 for Literal:")
    scores = []
    for i, row in sample.iterrows():
        print(f"Hindi: {row['Sentence']}")
        print(f"Output: {row['M2M_Baseline_Output']}")
        score = int(input("Score: "))
        scores.append(score)
    return (sum(scores) / n) * 100